# Machine Learning with scikit-learn

scikit-learn makes machine learning accessible without sacrificing power. Every algorithm (from logistic regression to random forests) shares the same three-method API: `fit()`, `predict()`, `score()`. Swap one model for another in a single line. Pipelines chain preprocessing and modelling so there's no risk of data leaking between train and test sets.

**What's inside:** the estimator API, classification, regression, clustering, preprocessing, pipelines, cross-validation, and hyperparameter tuning.

**Learn more:** [scikit-learn documentation](https://scikit-learn.org/stable/)

## Setup

In [ ]:
%pip install scikit-learn numpy pandas

## 1. The Estimator API

Every scikit-learn model follows the same interface regardless of its complexity.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# the same three calls work for every model
for Model in [LogisticRegression, DecisionTreeClassifier, RandomForestClassifier]:
    model = Model(random_state=42, max_iter=200)
    model.fit(X_train, y_train)
    print(f'{Model.__name__:<30} accuracy={model.score(X_test, y_test):.3f}')

## 2. Classification

### 2.1 Predictions and probabilities

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('predictions: ', y_pred[:10])
print('actual:      ', y_test[:10])

In [ ]:
# predict_proba gives the probability for each class
probs = model.predict_proba(X_test)
print('class probabilities (first 5 rows):')
print(np.round(probs[:5], 3))

### 2.2 Evaluation metrics

In [ ]:
from sklearn.metrics import classification_report

target_names = load_iris().target_names
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)

### 2.3 Feature importance

Tree-based models expose how much each feature contributed to the predictions.

In [ ]:
feature_names = load_iris().feature_names
importances = model.feature_importances_

for name, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    print(f'{name:<25} {imp:.4f}')

## 3. Regression

### 3.1 Linear, Ridge, and Lasso

Ridge adds L2 regularisation; Lasso adds L1 regularisation and can shrink coefficients to exactly zero (automatic feature selection).

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for Model, kwargs in [(LinearRegression, {}), (Ridge, {'alpha': 1.0}), (Lasso, {'alpha': 0.1})]:
    m = Model(**kwargs).fit(X_train, y_train)
    y_pred = m.predict(X_test)
    print(f'{Model.__name__:<20} R²={r2_score(y_test, y_pred):.3f}  '
          f'RMSE={mean_squared_error(y_test, y_pred, squared=False):.2f}')

In [ ]:
# Lasso zeros out less important coefficients
lasso = Lasso(alpha=1.0).fit(X_train, y_train)
feature_names = load_diabetes().feature_names

for name, coef in zip(feature_names, lasso.coef_):
    status = '(zeroed out)' if coef == 0 else ''
    print(f'{name:<5} {coef:8.3f}  {status}')

### 3.2 Regression metrics

In [ ]:
from sklearn.metrics import mean_absolute_error

model = Ridge(alpha=1.0).fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'R²:   {r2_score(y_test, y_pred):.4f}')          # 1.0 = perfect
print(f'MAE:  {mean_absolute_error(y_test, y_pred):.4f}') # avg absolute error
print(f'RMSE: {mean_squared_error(y_test, y_pred, squared=False):.4f}') # penalises large errors

## 4. Clustering

Clustering is unsupervised; no labels needed. The algorithm finds structure in the data itself.

### 4.1 K-Means

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

# adjusted Rand index: 1.0 = perfect match to true labels, 0.0 = random
print(f'adjusted Rand index: {adjusted_rand_score(y_true, labels):.4f}')

In [ ]:
# cluster centres
print('cluster centres:')
print(kmeans.cluster_centers_.round(2))

### 4.2 DBSCAN

DBSCAN discovers clusters of arbitrary shape and marks outliers as noise (label -1). No need to specify the number of clusters.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import numpy as np

X_scaled = StandardScaler().fit_transform(X)

db = DBSCAN(eps=0.3, min_samples=5).fit(X_scaled)
print(f'clusters found: {len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)}')
print(f'noise points:   {np.sum(db.labels_ == -1)}')

## 5. Preprocessing

Raw data rarely arrives in the right form. scikit-learn's transformers follow the same `fit` / `transform` pattern as estimators.

### 5.1 Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import numpy as np

data = np.array([[1, 200], [2, 400], [3, 300]], dtype=float)

# StandardScaler: zero mean, unit variance
print('StandardScaler:')
print(StandardScaler().fit_transform(data).round(4))

In [ ]:
# MinMaxScaler: scales to [0, 1]
print('MinMaxScaler:')
print(MinMaxScaler().fit_transform(data).round(4))

### 5.2 Encoding categorical features

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import numpy as np

categories = np.array([['red'], ['blue'], ['green'], ['red']])

# OneHotEncoder: creates a binary column per category
enc = OneHotEncoder(sparse_output=False)
print(enc.fit_transform(categories))
print(enc.categories_)

In [ ]:
# LabelEncoder: maps categories to integers (for target labels)
le = LabelEncoder()
print(le.fit_transform(['cat', 'dog', 'cat', 'bird']))
print(le.classes_)

### 5.3 Handling missing data

In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np

X_missing = np.array([[1, 2], [np.nan, 4], [5, np.nan], [7, 8]])

# fill missing values with the column mean
imp = SimpleImputer(strategy='mean')
print(imp.fit_transform(X_missing))

## 6. Pipelines

A `Pipeline` chains transformers and an estimator into a single object. The key benefit: `fit()` only ever sees training data, so there's no risk of test data leaking into preprocessing.

### 6.1 Basic pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=200)),
])

pipe.fit(X_train, y_train)
print(f'accuracy: {pipe.score(X_test, y_test):.4f}')

In [ ]:
# the pipeline predicts exactly like a standalone model
pipe.predict(X_test[:5])

### 6.2 ColumnTransformer: mixed data types

Apply different preprocessing to numeric and categorical columns simultaneously.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# toy dataset: age (numeric), city (categorical), income (numeric with missing)
X = np.array([
    [25, 'Austin', 50000],
    [32, 'Boston', 72000],
    [28, 'Austin', None ],
    [45, 'Denver', 91000],
    [36, 'Boston', 64000],
], dtype=object)
y = np.array([0, 1, 0, 1, 1])

numeric_cols     = [0, 2]   # age, income
categorical_cols = [1]      # city

preprocessor = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer()), ('scale', StandardScaler())]), numeric_cols),
    ('cat', OneHotEncoder(), categorical_cols),
])

pipe = Pipeline([('prep', preprocessor), ('model', RandomForestClassifier(random_state=42))])
pipe.fit(X, y)
print('fitted successfully')
print('transformed shape:', preprocessor.fit_transform(X).shape)

## 7. Model Selection

### 7.1 Cross-validation

A single train/test split can be lucky or unlucky. Cross-validation gives a more reliable estimate by training and evaluating on multiple different splits.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
import numpy as np

X, y = load_iris(return_X_y=True)

scores = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=5)
print('fold scores: ', scores.round(4))
print(f'mean: {scores.mean():.4f}  std: {scores.std():.4f}')

### 7.2 GridSearchCV: hyperparameter tuning

Exhaustively searches a grid of hyperparameter values using cross-validation to find the best combination.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 3, 5],
}

search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, n_jobs=-1)
search.fit(X, y)

print('best params: ', search.best_params_)
print(f'best score:  {search.best_score_:.4f}')

In [ ]:
# the best estimator is ready to use directly
best_model = search.best_estimator_
best_model.score(X, y)

### 7.3 Comparing multiple models with cross-validation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
import numpy as np

models = [
    ('Logistic Regression',   LogisticRegression(max_iter=200)),
    ('Decision Tree',         DecisionTreeClassifier()),
    ('Random Forest',         RandomForestClassifier(random_state=42)),
    ('Gradient Boosting',     GradientBoostingClassifier(random_state=42)),
    ('SVM',                   SVC()),
]

for name, model in models:
    scores = cross_val_score(model, X, y, cv=5)
    print(f'{name:<25} {scores.mean():.4f} ± {scores.std():.4f}')